# ORBIT: обучение модели мира с нуля / Training from scratch

Восстановленный temporal JEPA: два наблюдения → прогноз признаков будущего кадра → EMA teacher. Затем отдельно обучаются координатный probe и предиктор действий. Все сети начинают со случайных весов; координаты используются только для probe, который рисует прогноз и не выбирает действия.

This notebook executes the recovered training code in an empty temporary workspace. No pretrained weights are loaded. This is a small educational JEPA-inspired model, not Meta I-JEPA/V-JEPA. The default is the full recipe (2,400 visual, 800 probe and 5,000 action steps). CI also runs the full recipe. Set `ORBIT_NOTEBOOK_SMOKE=1` for an optional short execution check; its metrics are not a quality benchmark.

[Method, measured results and limitations](../FROM_SCRATCH.md) · [Preserved source](../ml/original-training/)

In [1]:
from pathlib import Path
import json, os, sys, tempfile
root = Path.cwd()
if not (root / 'planning').is_dir():
    root = root.parent
assert (root / 'ml/original-training/train.py').exists(), 'Run from this repository'
sys.path.insert(0, str(root))
from planning.original_training import train_pipeline
from planning.model import load_model, scenario
from planning.benchmark import evaluate
smoke = os.environ.get('ORBIT_NOTEBOOK_SMOKE') == '1'
print('SMOKE: execution check only' if smoke else 'FULL training from random initialization')

FULL training from random initialization


## 1. Обучить все компоненты / Train all components

The runner checks source hashes, copies only code and reports into an empty directory, and runs the unchanged original numerical code. Outputs remain under ignored `runs/`; repeated executions create new directories. The preserved original tests check stop-gradient, EMA, absence of future leakage and browser inference parity.

In [2]:
(root / 'runs').mkdir(exist_ok=True)
run_parent = Path(tempfile.mkdtemp(prefix='notebook-original-', dir=root / 'runs'))
output = run_parent / 'model'
report = train_pipeline(output, seed=44, action_seed=31005, smoke=smoke)
assert not report['pretrained_weights_read']
assert report['empty_training_workspace_verified']
print('New bundle:', report['browser_model_id'])
print('Visual prediction error:', round(report['visual_position_error_px'], 4), 'pixels')

01_visual_train


02_probe_evaluate


03_action_train


04_action_evaluate


05_browser_export


06_visual_tests


07_action_tests


08_browser_parity


Complete: model; historical-scene hit rate 0.7207


New bundle: d7aff1ecd184
Visual prediction error: 0.7811 pixels


## 2. Проверить планирование / Use the newly trained model

The planner receives two observed images and the goal image. It compares all 27 action sequences in feature space. The coordinate probe is used only to display the predicted path.

In [3]:
model = load_model(model_dir=output)
scene = scenario(37)
best, costs, predictions = model.rank(scene['frames'], scene['goal_frame'])
from planning.model import PLANS, rollout
print('Actions:', PLANS[best].astype(int).tolist())
print('Predicted positions:', model.decode(predictions[best]).round(3).tolist())
print('Actual positions:', rollout(scene['p1'], scene['v'], PLANS[best]).round(3).tolist())

Actions: [-1, 0, -1]
Predicted positions: [[0.411, 0.886], [0.248, 0.8], [0.196, 0.715]]
Actual positions: [[0.378, 0.864], [0.266, 0.773], [0.085, 0.681]]


## 3. Новый тест / Fresh evaluation

The following cell actually evaluates 256 scenes, separate from training and validation. This short sample is not the published 3,072-scene benchmark. Random control and a privileged exact-physics oracle use the same scenes.

In [4]:
result = evaluate(model, seeds=[88771], per_seed=256)
for name in ['model', 'random', 'no_push', 'oracle']:
    row = result['aggregate'][name]
    print(f"{name}: {100*row['hit_rate']:.1f}% hits; {row['mean_distance_px']:.3f} px")

model: 73.0% hits; 0.869 px
random: 19.9% hits; 4.422 px
no_push: 23.4% hits; 3.491 px
oracle: 100.0% hits; 0.198 px


## 4. Что воспроизвелось / Reproducibility evidence

Byte identity with historical Linux weights is not guaranteed on a different OS or library stack. Below are the actual comparisons, not assumed matches. The supplied original weights match the tensors used in the game; local newly trained weights can differ. Full logs, configuration and checkpoints are in this run directory.

The published comparison of temporal learning and same-state VICReg uses different complete recipes, so it cannot isolate the causal effect of the encoder objective alone.

In [5]:
for name, row in report['historical_weight_comparison'].items():
    print(name, 'matches historical bytes:', row['matches_supplied_reference_bytes'])
reference = json.loads((root / 'experiments/temporal-44/training.json').read_text())
if not smoke:
    print('Matches published local full run:', report['artifacts_sha256'] == reference['artifacts_sha256'])
print('Environment:', report['environment'])

pretrained/model.npz matches historical bytes: False
pretrained/probe.npz matches historical bytes: False
action_jepa/predictor.npz matches historical bytes: False
Matches published local full run: True
Environment: {'python': '3.12.14', 'numpy': '2.1.3', 'torch': '2.5.1', 'platform': 'macOS-27.2.1-arm64-arm-64bit', 'cpu': 'Apple M2', 'architecture': 'arm64', 'device': 'cpu', 'threads': 2}
